# Step 1: Persona Induction & Validation

We induce 6 personas and validate that they produce distinguishable activation
signatures. 5 personas are positioned along the **assistant axis** (Lu et al., 2026)
from deep roleplay (-1.0) to full assistant (+1.0), plus a base model control.

| Position | Persona | Description |
|----------|---------|-------------|
| N/A | Base Model | Raw text completion, no chat template |
| -1.0 | Deep Roleplay | Fully embodied character, enigmatic |
| -0.5 | Mild Roleplay | Character with personality, still functional |
|  0.0 | Neutral | Minimal system prompt |
| +0.5 | Mild Assistant | Helpful AI, not performative |
| +1.0 | Full Assistant | Maximally helpful RLHF-aligned assistant |

Validation: collect mean activations on neutral prompts, compare per-layer
cosine similarity between all persona pairs.

In [ ]:
import torch
from nnsight import LanguageModel

from persona_steering.config import GEMMA_2_9B
from persona_steering.personas import PersonaInducer, load_all_personas, load_axis_personas
from persona_steering.utils import get_device, ensure_output_dirs, save_pickle
from persona_steering.config import ACTIVATIONS_DIR

ensure_output_dirs()
device = get_device()
print(f"Device: {device}")

In [ ]:
# Load model
config = GEMMA_2_9B
model = LanguageModel(config.hf_id, device_map="auto", torch_dtype=torch.float16)
tokenizer = model.tokenizer
print(f"Loaded {config.name}")

In [ ]:
# Load all 6 personas
all_personas = load_all_personas()
axis_personas = load_axis_personas()

print(f"All personas ({len(all_personas)}):")
for p in all_personas:
    pos = f"{p.position:+.1f}" if p.position is not None else "off-axis"
    print(f"  {p.name} [{pos}]: {p.description[:70].strip()}...")

print(f"\nAxis personas ({len(axis_personas)}):")
for p in axis_personas:
    print(f"  {p.position:+.1f}  {p.name}")

In [ ]:
# Collect activations for each persona on neutral prompts
inducer = PersonaInducer(model, tokenizer)
layers = config.extraction_layers

neutral_prompts = [
    "What is the capital of France?",
    "Explain how photosynthesis works.",
    "What are the benefits of exercise?",
    "Describe the water cycle.",
    "How does a computer processor work?",
    "What causes seasons on Earth?",
    "Explain supply and demand.",
    "What is the scientific method?",
    "How do vaccines work?",
    "Describe the structure of DNA.",
]

for persona in all_personas:
    inducer.collect_activations(persona, neutral_prompts, layers)

save_pickle(inducer._cached_activations, ACTIVATIONS_DIR / "persona_activations.pkl")

In [ ]:
# Validate: pairwise cosine similarity heatmap across all personas
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

names = [p.name for p in all_personas]
n = len(all_personas)

# Average cosine sim across layers for each pair
sim_matrix = np.ones((n, n))
for i in range(n):
    for j in range(i + 1, n):
        sims = inducer.validate_persona(all_personas[i], all_personas[j], layers)
        mean_sim = np.mean(list(sims.values()))
        sim_matrix[i, j] = mean_sim
        sim_matrix[j, i] = mean_sim

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(sim_matrix, annot=True, fmt=".3f", xticklabels=names,
            yticklabels=names, cmap="RdYlBu_r", vmin=0.85, vmax=1.0,
            ax=ax, square=True)
ax.set_title("Persona Activation Similarity (mean across layers)")
plt.tight_layout()
plt.savefig("../outputs/figures/persona_validation.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Layer-wise similarity for axis personas (should show gradient)
fig, ax = plt.subplots(figsize=(10, 5))

# Compare each adjacent pair on the axis
for i in range(len(axis_personas) - 1):
    pa, pb = axis_personas[i], axis_personas[i + 1]
    sims = inducer.validate_persona(pa, pb, layers)
    layer_ids = sorted(sims.keys())
    sim_vals = [sims[l] for l in layer_ids]
    label = f"{pa.name} vs {pb.name}"
    ax.plot(layer_ids, sim_vals, marker='o', markersize=3, label=label)

# Also compare extremes
sims = inducer.validate_persona(axis_personas[0], axis_personas[-1], layers)
layer_ids = sorted(sims.keys())
ax.plot(layer_ids, [sims[l] for l in layer_ids], marker='s', markersize=3,
        linestyle='--', label=f"{axis_personas[0].name} vs {axis_personas[-1].name}", color='red')

ax.set_xlabel("Layer")
ax.set_ylabel("Cosine Similarity")
ax.set_title("Layer-wise Persona Similarity Along Assistant Axis")
ax.legend(loc="lower left", fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("../outputs/figures/axis_persona_layerwise.png", dpi=150, bbox_inches="tight")
plt.show()
print("Persona validation complete.")